**Now we'll create the second Gold table: customer-level sales analytics.**

```text
SILVER
   │
   │ customer aggregation
   ↓
GOLD_CUSTOMERS
   │
   ├── Total Orders
   ├── Total Spend
   └── Last Order Date
```

### Create the Gold table

In [0]:
%sql
CREATE OR REPLACE TABLE retail_lakehouse.gold.gold_customers
USING DELTA
AS

SELECT
    customer_id,

    COUNT(DISTINCT order_id) AS total_orders,

    ROUND(
        SUM(quantity * unit_price),
        2
    ) AS total_spend,

    MAX(order_date) AS last_order_date

FROM retail_lakehouse.silver.orders

GROUP BY customer_id;

num_affected_rows,num_inserted_rows


### Validate the table

In [0]:
%sql
SELECT *
FROM retail_lakehouse.gold.gold_customers
ORDER BY total_spend DESC
LIMIT 10;

customer_id,total_orders,total_spend,last_order_date
1079,92,115518.46,2026-08-27
1097,92,112293.67,2026-08-27
1113,93,108505.70,2026-08-27
1107,83,107683.04,2026-08-27
1140,79,104318.57,2026-08-27
1011,79,103179.28,2026-08-27
1131,81,103021.67,2026-08-27
1014,81,102850.51,2026-08-27
1051,78,101894.34,2026-08-27
1068,83,100113.66,2026-08-27


### Find high-value customers

In [0]:
%sql
SELECT
    customer_id,
    total_orders,
    total_spend,
    last_order_date
FROM retail_lakehouse.gold.gold_customers
WHERE total_spend >= 10000
ORDER BY total_spend DESC;

customer_id,total_orders,total_spend,last_order_date
1079,92,115518.46,2026-08-27
1097,92,112293.67,2026-08-27
1113,93,108505.70,2026-08-27
1107,83,107683.04,2026-08-27
1140,79,104318.57,2026-08-27
1011,79,103179.28,2026-08-27
1131,81,103021.67,2026-08-27
1014,81,102850.51,2026-08-27
1051,78,101894.34,2026-08-27
1068,83,100113.66,2026-08-27


### Customer segmentation

simple business segment:

In [0]:
%sql
SELECT
    customer_id,
    total_orders,
    total_spend,
    last_order_date,

    CASE
        WHEN total_spend >= 20000 THEN 'VIP'
        WHEN total_spend >= 10000 THEN 'HIGH_VALUE'
        WHEN total_spend >= 5000 THEN 'MEDIUM_VALUE'
        ELSE 'REGULAR'
    END AS customer_segment

FROM retail_lakehouse.gold.gold_customers;

-- For this project i would keep segmentation out of the physical table and add it later if required. 
-- The core Gold table should remain simple.

customer_id,total_orders,total_spend,last_order_date,customer_segment
1117,61,72569.21,2026-08-27,VIP
1168,68,61440.84,2026-08-27,VIP
1025,77,87342.92,2026-08-27,VIP
1004,60,69146.86,2026-08-27,VIP
1027,69,80435.22,2026-08-27,VIP
1124,71,88817.33,2026-08-27,VIP
1130,68,76757.87,2026-08-27,VIP
1133,69,73597.00,2026-08-27,VIP
1177,70,82609.75,2026-08-27,VIP
1126,82,88384.65,2026-08-27,VIP


In [0]:
%sql
-- Validation
SELECT *
FROM retail_lakehouse.gold.gold_customers
ORDER BY total_spend DESC
LIMIT 10;

customer_id,total_orders,total_spend,last_order_date
1079,92,115518.46,2026-08-27
1097,92,112293.67,2026-08-27
1113,93,108505.70,2026-08-27
1107,83,107683.04,2026-08-27
1140,79,104318.57,2026-08-27
1011,79,103179.28,2026-08-27
1131,81,103021.67,2026-08-27
1014,81,102850.51,2026-08-27
1051,78,101894.34,2026-08-27
1068,83,100113.66,2026-08-27
